In [1]:
import sys, os
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('backbone')))) # to import file from under same-level directory
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('utils'))))
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('neck'))))

from backbone.convnext_se.convnext_se import ConvNextV1, Stage, Stem,  DetectorBackbone
from utils.conv_2d import adjust_padding_for_strided_output, DepthWiseSepConv
from utils.stochastic_depth_drop import create_linear_p, create_uniform_p
from neck.convnext_se_fpn import FPN

import os
import glob
import PIL
from PIL import Image
from PIL import ImageDraw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
import torchinfo
import albumentations
import pprint


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.23). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# example image to use
ex_img = Image.open("/work/dataset/CULane/driver_23_30frame/05151640_0419.MP4/00000.jpg")
print("original image size: ", ex_img.size)
# detector also will have much smaller image scale than original image
ex_img_resize = ex_img.resize([672, 224])
ex_img = torch.tensor(np.array(ex_img_resize)).unsqueeze(0).permute(0, 3, 1, 2).float().to('cuda')
print(ex_img.shape)

original image size:  (1640, 590)
torch.Size([1, 3, 224, 672])


In [3]:
# example label to use
with open('/work/dataset/CULane/processed_labels/driver_23_30frame/05151640_0419.MP4/00000.lines.txt', 'r') as fr:
    ex_label = fr.readlines()
ex_label = [ex.strip() for ex in ex_label]

point_label = []
for i in range(len(ex_label)):
    lane_tokens = ex_label[i].split(' ')
    lane = []
    for j in range(0, len(lane_tokens), 2):
        lane.append((int(lane_tokens[j]), int(lane_tokens[j+1])))
    point_label.append(lane)

point_label = np.array(point_label)
print("label shape: ", point_label.shape) # number of lanes, number of points, x,y in each point
point_label

label shape:  (3, 35, 2)


array([[[ 240,  590],
        [ 257,  580],
        [ 275,  570],
        [ 292,  560],
        [ 309,  550],
        [ 327,  540],
        [ 344,  530],
        [ 361,  520],
        [ 379,  510],
        [ 396,  500],
        [ 413,  490],
        [ 431,  480],
        [ 448,  470],
        [ 467,  460],
        [ 484,  450],
        [ 502,  440],
        [ 520,  430],
        [ 538,  420],
        [ 557,  410],
        [ 575,  400],
        [ 593,  390],
        [ 612,  380],
        [ 630,  370],
        [ 649,  360],
        [ 667,  350],
        [ 686,  340],
        [ 704,  330],
        [ 722,  320],
        [ 741,  310],
        [ 759,  300],
        [ 778,  290],
        [  -1,  280],
        [  -1,  270],
        [  -1,  260],
        [  -1,  250]],

       [[1146,  590],
        [1133,  580],
        [1120,  570],
        [1108,  560],
        [1097,  550],
        [1084,  540],
        [1073,  530],
        [1060,  520],
        [1049,  510],
        [1037,  500],
        

In [4]:
dp_list, dp_mode = create_linear_p([3,3,9,3], 'batch', 0.25) # creates linearly decaying stochastic depth drop rate and mode

detector_encoder = DetectorBackbone(num_blocks=[3,3,9,3], input_channels=3, stem_kersz=(4,4), stem_stride=(4,4), 
                                   img_hw=[(56, 168), (28, 84), (14, 42), (7,21)], main_channels=[48, 96, 192, 384], expansion_dim=[48*4, 96*4, 192*4, 384*4], 
                                   kernel_sz=[(7,7)]*4, stride=[(1,1)]*4, padding=['same']*4, dilation=[1]*4, groups=[1]*4, droprate=dp_list, drop_mode=dp_mode, 
                                   use_se=[True]*4, squeeze_ratio=16, transition_kersz=[-1, (1,1), (1,1), (1,1)], transition_stride=[-1, (2,2), (2,2), (2,2)], norm_mode='layer_norm', device='cuda')

# fpn has the structure that goes through 0 to 3 stage output, we can use corresponding (28, 84), (14, 42), (7, 21) stage output,
# or we can use all stages output to constructure fpn structure.
backbone_o = detector_encoder(ex_img)
for i, stage in enumerate(backbone_o):
    print(f"each stage output {i}: ", stage.shape)

torchinfo.summary(detector_encoder, (1, 3, 224, 672))

each stage output 0:  torch.Size([1, 48, 56, 168])
each stage output 1:  torch.Size([1, 96, 28, 84])
each stage output 2:  torch.Size([1, 192, 14, 42])
each stage output 3:  torch.Size([1, 384, 7, 21])


Layer (type:depth-idx)                             Output Shape              Param #
DetectorBackbone                                   [1, 48, 56, 168]          --
├─Stem: 1-1                                        [1, 48, 56, 168]          --
│    └─Conv2d: 2-1                                 [1, 48, 56, 168]          2,352
├─ModuleList: 1-2                                  --                        --
│    └─Stage: 2-2                                  [1, 48, 56, 168]          --
│    │    └─ModuleList: 3-1                        --                        2,780,793
│    └─Stage: 2-3                                  [1, 96, 28, 84]           --
│    │    └─ModuleList: 3-2                        --                        1,621,170
│    └─Stage: 2-4                                  [1, 192, 14, 42]          --
│    │    └─ModuleList: 3-3                        --                        5,153,580
│    └─Stage: 2-5                                  [1, 384, 7, 21]           --
│    │    └

In [5]:
# class FPN(torch.nn.ModuleList):
#     def __init__(self, nstage_to_use, bidirectional, in_channels, out_channels, img_hw, downsample_kersz, upsample_kersz, downsample_stride, upsample_stride, 
#                  downsample_dilation, upsample_dilation, downsample_groups, upsample_groups, upsample_padding, upsample_out_padding, norm_mode, device, dtype):

fpn = FPN(nstage_to_use=4, bidirectional=True, in_channels=[48, 96, 192, 384], out_channels=224, img_hw=[(56, 168), (28, 84), (14, 42), (7, 21)],
          downsample_kersz=[(7,7), (7,7), (7,7)], upsample_kersz=[(7,7), (7,7), (7,7)], downsample_stride=[(2,2), (2,2), (2,2)], upsample_stride=[(2,2), (2,2), (2,2)],
          downsample_dilation=[1,1,1], upsample_dilation=[1,1,1], downsample_groups=[1,1,1], upsample_groups=[1,1,1], 
          upsample_padding=[(3,3), (3,3), (3,3)], upsample_out_padding=[(1,1), (1,1), (1,1)], norm_mode='layer_norm', device='cuda', dtype=None)
fpn_output = fpn(backbone_o)
for i in range(len(fpn_output)):
    print(f"{i}th fpn output shape: ", fpn_output[i].shape)

0th fpn output shape:  torch.Size([1, 224, 7, 21])
1th fpn output shape:  torch.Size([1, 224, 14, 42])
2th fpn output shape:  torch.Size([1, 224, 28, 84])
3th fpn output shape:  torch.Size([1, 224, 56, 168])


In [6]:
input_ = fpn_output[-1].repeat(2, 1, 1, 1)
print("size of input_:", input_.shape)
roi_o = torchvision.ops.roi_align(input_, boxes=[torch.tensor([[2.0, 1.0, 4.0, 5.0],
                                                              [3.0, 2.0, 4.0, 5.0],
                                                            [5.0, 6.0, 7.0, 9.0]],
                                                              device='cuda'),
                                                 torch.tensor([[2.0, 1.0, 4.0, 5.0]],
                                                              device='cuda')], 
                                  output_size=(10, 10), spatial_scale=1/32, sampling_ratio=-1)
print("output of roi alignment: ", roi_o.shape)

size of input_: torch.Size([2, 224, 56, 168])
output of roi alignment:  torch.Size([4, 224, 10, 10])


In [19]:
a = torch.ones((100, 135))
index = torch.randint(low=0, high=100, size=(30, )) # samples 30 from 0~100 int range
a[index, :].shape
sorted_, indics = torch.sort(index)
sorted_
b = torch.ones((32, 140)) * -2
torch.clip(b+1, min = 0, max=1280) # plus are broadcasted
b.int()

tensor([[-2, -2, -2,  ..., -2, -2, -2],
        [-2, -2, -2,  ..., -2, -2, -2],
        [-2, -2, -2,  ..., -2, -2, -2],
        ...,
        [-2, -2, -2,  ..., -2, -2, -2],
        [-2, -2, -2,  ..., -2, -2, -2],
        [-2, -2, -2,  ..., -2, -2, -2]], dtype=torch.int32)

In [ ]:
# head of the network: 
class CLRHead(torch.nn.Module):
    def __init__(self, in_channels, nstages, n_lanes, input_img_size, context_sz, roi_sz,
                 num_samples, num_points, y_prior,
                 c1d_ouc, c1d_ker, intp_h, intp_w, device, dtype=None):
        super(CLRHead, self).__init__()
        self.in_channels = in_channels      # number of channels from FPN output
        self.nstages = nstages              # number of stages from FPN
        self.n_lanes = n_lanes                  # total number of lanes
        self.input_img_size = input_img_size # img size that boxes are defined (in this case, points)
        self.context_sz = context_sz # context size of the point (from the point as center of box, it determines the amount of nearby pixels)
        self.roi_sz = roi_sz # output of roi aligment feature size (given as list for each pyramid level)
        self.num_samples = num_samples # number of sampled points in roi region
        self.num_points = num_points # number of total points to predict
        
        # y_prior: (y1, y2, step) -> y1 > y2 (descending order), step: positive pixel difference between y points
        self.y_prior = range(y_prior[0], y_prior[1], -y_prior[2]) # y coordinate of the points in the image (y coordinate are defined in fixed height)
        
        self.device = device            # 'cuda' or 'cpu'
        self.dtype = dtype              # default to None
        
        self.c1d_ouc = c1d_ouc
        self.c1d_ker = c1d_ker
        
        self.intp_h = intp_h # target interpolate height
        self.intp_w = intp_w # target interpolate width
        
        # conv1d for learning sequence characteristic of local features (around sampled points)
        self.conv_block = torch.nn.Sequential([nn.Conv1d(in_channels=in_channels, out_channels=self.c1d_ouc, kernel_size=self.c1d_ker, device=device, dtype=dtype),
                            nn.ReLU(),
                            nn.Conv1d(in_channels=self.c1d_ouc, out_channels=self.c1d_ouc, kernel_size=self.c1d_ker, device=device, dtype=dtype)])
        
        # self attention for feature learning between local point features        
        self.q = nn.Linear(c1d_ouc, c1d_ouc, device=device, dtype=dtype)
        self.k = nn.Linear(c1d_ouc, c1d_ouc, device=device, dtype=dtype)
        self.v = nn.Linear(c1d_ouc, c1d_ouc, device=device, dtype=dtype)
        self.o = nn.Linear(c1d_ouc, c1d_ouc, device=device, dtype=dtype)
        self.softmax = nn.Softmax(dim=-1)
        
        # this is for adjusting number of channels in interpolated global feature map
        self.pointwise_conv = torch.nn.Conv2d(in_channels=self.in_channels, out_channels=self.c1d_ouc,
                                              kernel_size=(1, 1), stride=(1, 1), padding=(0, 0), device=device, dtype=dtype)
        
        # this is for adjusting dimension size between local and global feature map
        self.a2l = torch.nn.Linear(self.intp_h * self.intp_w, self.c1d_ouc, device=device, dtype=dtype)
        
        # point regressor and classifier does not share weights
        # lane point regressor branch (point position regressor)
        self.p_reg = torch.nn.Sequential([torch.nn.Linear(self.num_samples, self.num_points, device=device, dtype=dtype),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(self.num_points, self.num_points, device=device, dtype=dtype)])
        
        # lane point classifier branch(point validation classifier)
        self.p_cls = torch.nn.Sequential([torch.nn.Linear(self.num_samples, self.num_points, device=device, dtype=dtype),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(self.num_points, self.num_points, device=device, dtype=dtype)])

    # reg_o shape: (b, num_point) -> normalized x coordinate (0~1) for each y coordinates (num_point is same as the number of y coordinates in self.y_prior)
    # output shape: (b, num_point, 4) tensor
    # need to consider, cls_o return invalid point for that location (-1), then how we count this point?
    def update_box_loc(self, reg_o):
        # sample as much as self.num_samples from regression_output (consists of (b, num_point))
        x_sample, _ = torch.sort(torch.randint(low=0, high=reg_o.shape[1], size=(self.num_samples, )))
        # (batch, num_points) -> (batch, num_samples, 1)
        sampled_x = (reg_o[:, x_sample] * self.input_img_size[0]) # randomly sample point from output (for example, 70 from total 140)
        sampled_y = (self.y_prior.repeat(reg_o.shape[0], 1)[:, x_sample]) # (batch, num_samples, 1)
        
        # locations are cliped as min=0, max=self.input_img_size[0] or [1]
        # context boundaries does not exceed image boundaries
        x1 = torch.clip(sampled_x - self.context_sz, min=0, max=self.input_img_size[1]).unsqueeze(-1).int()
        y1 = torch.clip(sampled_y - self.context_sz, min=0, max=self.input_img_size[0]).unsqueeze(-1).int()
        x2 = torch.clip(sampled_x + self.context_sz, min=0, max=self.input_img_size[1]).unsqueeze(-1).int()
        y2 = torch.clip(sampled_y + self.context_sz, min=0, max=self.input_img_size[0]).unsqueeze(-1).int()
        
        # (b, num_samples) -> (stacked) -> (b, num_samples, 4)
        box_loc = torch.stack([x1, y1, x2, y2], dim=-1)
        return box_loc
        
    # x shape: (b, num_samples, c1d_ouc)
    def self_attention(self, x):
        query = self.q(x)
        key = self.k(x)
        value = self.v(x)
        scale = 1.0 / (self.c1d_ouc ** 0.5)
        
        attn_map = query @ key.transpose(-2, -1) # (b, num_samples, num_samples)
        attn_dist = self.softmax(attn_map * scale) # (b, num_samples, num_samples)
        attn_out = attn_dist @ value # (b, num_samples, c1d_ouc)
        attn_out = self.o(attn_out)
        return attn_out
        
    # pbox_init: initial boxes location for roi align (in case of clrnet, they use randomly uniformly sampled points from the image)
    # pbox must be learnable queries that represent the initial location of point distribution (but this is represented as box with context size)
    # ex) x shape: [(1, 224, 7, 21), (1, 224, 14, 42), (1, 224, 28, 84), (1, 224, 56, 168)]
    def forward(self, x, pbox_init):
        regressor_output = [] # point regression output by head
        classifier_output = [] # classifier output by head
        
        box_loc = pbox_init # pbox_init -> [torch.tensor([[x1, y1, x2, y2] * num_samples])] * batch -> (b, num_samples, 4) shape 
        for p_i, pyramid in enumerate(x, 0): # from top to bottom
            nb, nc = pyramid.shape[0], pyramid.shape[1] # number of batch, number of channels
            p_h, p_w = pyramid.shape[2], pyramid.shape[3] # height, width of feature in pyramid
            rois = torchvision.ops.roi_align(pyramid, boxes=box_loc, output_size=self.roi_sz[p_i],
                                             spatial_scale=(self.input_img_size[0]/p_h), sampling_ratio=-1)
            
            # (b * num_samples, c, roi_sz[0], roi_sz[1]) -> (b * num_samples, c, 1, 1)
            rois = torch.nn.functional.avg_pool2d(rois, kernel_size=(self.roi_sz[p_i][0], self.roi_sz[p_i][1])) # main information carries through channel direction
            rois = torch.flatten(rois, start_dim=1, end_dim=-1) # (b * num_samples, c, 1, 1) -> (b * num_samples, c)
            
            # (b * num_samples, c) -> (b, num_samples, c)
            rois = rois.reshape(nb, -1, nc)
            # (b, num_samples, c) -> (b, c, num_samples)
            rois = torch.swapaxes(rois, 1, 2)
            # 1d convolution layer -> learns relationship between point representations locally
            rois = self.conv_block(rois)
            # (b, c1d_ouc, num_samples) -> (b, num_samples, c1d_ouc)
            rois = torch.swapaxes(rois, 1, 2)
            # self attention between extracted roi regions -> learns relationship between point representations glbally
            rois = self.self_attention(rois)
            
            
            # (b, c, h, w) -> (b, c, h_i, w_i)
            rescale_py = torch.nn.functional.interpolate(pyramid, size=(self.intp_h, self.intp_w),
                                                      mode='nearest')
            # (b, c, h_i, w_i) -> (b, c1d_ouc, h_i, w_i)
            rescale_py = self.pointwise_conv(rescale_py)
            # (b, c1d_ouc, h_i, w_i) -> (b, c1d_ouc, h_i * w_i)
            rescale_py = torch.flatten(rescale_py, start_dim=2, end_dim=-1)
            
            scale = 1.0 / (rois.shape[-1]**0.5)
            # attention between local and global -> (b, num_samples, h_i * w_i)
            attn_prob = self.softmax((rois @ rescale_py) * scale)
            
            # (b, num_samples, h_i * w_i) -> (b, num_samples, c1d_ouc)
            attn = self.a2l(attn_prob)
            # residual connection (local region of interest feature is reinforced by global feature)
            rois = attn + rois
            # (b, num_samples, c1d_ouc) -> (b, num_samples, 1) -> (b, num_samples)
            rois = torch.nn.functional.avg_pool1d(rois, kernel_size=rois.shape[-1])
            rois = rois.reshape(rois.shape[0], rois.shape[1])
            
            # (b, num_samples) -> (b, num_point)
            reg_o = self.p_reg(rois) # regressor output
            cls_o = self.p_cls(rois) # classifier output
            
            # update box_loc location
            box_loc = self.update_box_loc(reg_o, cls_o)
            
            regressor_output.append(reg_o)
            classifier_output.append(cls_o)

        return regressor_output, classifier_output

SyntaxError: invalid syntax (2880484033.py, line 67)

In [ ]:
class CLRNet(torch.nn.Module):
    def __init__(self, detector_encoder, fpn, clr_head):
        super().__init__()
        self.detector_encoder = detector_encoder
        self.fpn = fpn
        self.clr_head = clr_head
    
    def forward(self, x):
        pass

In [ ]:
# retinanet network head (anchors boxes are used)
class RetinaHead(torch.nn.Module):
    def __init__(self, in_channels, num_cls, num_anchors, nstages, branch_len, branch_kersz, branch_dilation, branch_groups, device, dtype):
        super().__init__()
        self.in_channels = in_channels       # number of input channels for head
        self.num_cls = num_cls               # number of class for classification branch
        self.nstages = nstages               # number of stages of input
        self.branch_len = branch_len         # number of iterations for head branch
        self.num_anchors = num_anchors       # number of anchors
        
        self.branch_kersz = branch_kersz        # kernel size in each branch convolution
        self.branch_dilation = branch_dilation  # dilation value (ratio of expansion) in each branch convolution
        self.branch_groups = branch_groups      # number of groups in each branch convolution
        
        self.device = device
        self.dtype = dtype
        
        # weight of the regression branches all shared across stages
        self.regression_branch = []
        for i in range(self.branch_len):
            self.regression_branch.append(torch.nn.Conv2d(in_channels=self.in_channels, out_channels=self.in_channels, kernel_size=self.branch_kersz, stride=(1,1), 
                                                          padding='same', dilation=self.branch_dilation, groups=self.branch_groups, device=self.device, dtype=self.dtype))
            self.regression_branch.append(torch.nn.ReLU())
        self.regression_branch = torch.nn.ModuleList(self.regression_branch)
            
        # weight of the classification branches all shared across stages
        self.cls_branch = []
        for i in range(self.branch_len):
            self.cls_branch.append(torch.nn.Conv2d(in_channels=self.in_channels, out_channels=self.in_channels, kernel_size=self.branch_kersz, stride=(1,1),
                                                   padding='same', dilation=self.branch_dilation, groups=self.branch_groups, device=self.device, dtype=self.dtype))
            self.cls_branch.append(torch.nn.ReLU())
        self.cls_branch = torch.nn.ModuleList(self.cls_branch)
        
        # (center_x, center_y, width, height) normalized version
        self.regression_out = torch.nn.Conv2d(in_channels=self.in_channels, out_channels=4 * self.num_anchors, kernel_size=(1,1), stride=(1,1),
                                              padding='same', dilation=1, groups=1, device=self.device, dtype=self.dtype)
        
        # number of classes * self.num_anchors for all tiled location
        self.cls_out = torch.nn.Conv2d(in_channels=self.in_channels, out_channels=self.num_cls * self.num_anchors, kernel_size=(1,1), stride=(1,1),
                                       padding='same', dilation=1, groups=1, device=self.device, dtype=self.dtype)
        self.cls_act = torch.nn.Softmax2d()
    
    # example fpn output): [(224,7,21), (224,14,42), (224,28,84), (224,56,168)]
    def forward(self, fpn_o):
        reg_x = fpn_o
        cls_x = fpn_o
        
        reg_o = [] # list of regression output
        cls_o = [] # list of clssification output
        
        for fpn_stage_o in fpn_o: # for each stage output corresponding to fpn_o
            reg_x = fpn_stage_o
            cls_x = fpn_stage_o
            for i in range(self.branch_len):
                reg_x = self.regression_branch[2*i](reg_x)
                reg_x = self.regression_branch[2*i+1](reg_x)
                
                cls_x = self.cls_branch[2*i](cls_x)
                cls_x = self.cls_branch[2*i+1](cls_x)
            
            reg_o.append(self.regression_out(reg_x))
            cls_o.append(self.cls_act(self.cls_out(cls_x)))
        
        return reg_o, cls_o

In [ ]:
vis_img = Image.open('/work/dataset/CULane/driver_193_90frame/06061052_0851.MP4/00000.jpg')
vis_img.show()
with open('/work/dataset/CULane/driver_193_90frame/06061052_0851.MP4/00000.lines.txt', 'r') as fr:
    vis_img_gt = fr.readlines()
for i in range(len(vis_img_gt)):
    vis_img_gt[i] = vis_img_gt[i].strip()
    

def draw_lanes(image, labels):
    color_list = ['red', 'blue', 'yellow', 'black']
    draw = ImageDraw.Draw(image)
    lines = []
    for p_set in labels:
        points = []
        tokens = p_set.split(' ')
        for token in tokens:
            points.append(float(token))
        lines.append(points)
    
    lines_p = []
    for line in lines:
        t = []
        for i in range(0, len(line), 2):
            t.append((line[i], line[i+1])) # pair x, y point locations
        lines_p.append(t)
    
    for lane_index, line in enumerate(lines_p, 0):
        for i in range(len(line)-1):
            draw.line((line[i][0], line[i][1], line[i+1][0], line[i+1][1]), fill=color_list[lane_index], width=5)
    
    return image

lane_img = draw_lanes(vis_img, vis_img_gt)
lane_img.show()